In [2]:
import os
import pandas as pd
import numpy as np

In [ ]:
from Settings import Settings

# ADNI

In [2]:
data_path = Settings["input_path"]
folder_name = 'MNI152lin_T1_1mm_brain'

In [3]:
image_data_file = pd.read_csv(f"{data_path}/zipped/ADNI_1_1_07_2025.csv")
image_data_file = image_data_file.drop_duplicates(subset = [
    "Subject", "Group", "Sex", "Age",	"Visit", "Modality", "Description", "Acq Date"
])
image_data_file = image_data_file.rename(columns = {
    "Sex" : "Sex_i",
    "Age" : "Age_i"
})
image_data_sorted = image_data_file.sort_values(
    by = ["Subject", "Acq Date"]
).reset_index()
image_data_sorted.shape

(2658, 13)

In [4]:
subject_data_file = pd.read_csv(f"{data_path}/zipped/idaSearch_1_07_2025 (1).csv")
subject_data_file["Description"] =  [x.split("<-")[0] for x in subject_data_file["Description"]]
subject_data_file = subject_data_file.drop_duplicates()
subject_data_sorted = subject_data_file.sort_values(
    by = ["Subject ID", "Age"]
).reset_index()
subject_data_sorted.shape

(2658, 10)

In [5]:
data_concat = pd.concat(
    [image_data_sorted, subject_data_sorted], 
    axis = 1
)

In [6]:
data_concat["check1"] = np.where(
    data_concat["Subject"] == data_concat["Subject ID"], 
    True, False
)
data_concat["check1"].value_counts()

check1
True    2658
Name: count, dtype: int64

In [7]:
data_concat = data_concat[[
    "Image Data ID", "Subject", "Group", "Sex", "Age", "Visit",
    "MMSE Total Score", "Global CDR", "Acq Date"
]]

In [8]:
data_concat["Group"].value_counts()

Group
MCI    1414
CN      776
AD      468
Name: count, dtype: int64

In [9]:
data_concat["Image_name"] = data_concat["Subject"] + "__" + data_concat["Image Data ID"]
data_concat["filepath"] = data_path + '/' + folder_name + "/Processed/" + data_concat["Image_name"] + ".nii.gz"

In [10]:
data_concat["Group"].value_counts()

Group
MCI    1414
CN      776
AD      468
Name: count, dtype: int64

In [11]:
df_check = data_concat.groupby(["Subject"])["Group"].apply(list).reset_index()
df_check["len"] = [len(list(set(x))) for x in df_check["Group"]]
df_check[df_check["len"] == 2]

,Subject,Group,len


In [12]:
df = data_concat[data_concat["Group"].isin(["AD","CN"])]

In [13]:
df = df[df["Visit"] == "sc"]

In [14]:
df= df.drop_duplicates(subset = ['Subject'])

In [15]:
df.shape

(313, 11)

In [16]:
df["Group"].value_counts()

Group
CN    168
AD    145
Name: count, dtype: int64

In [71]:
df.head()

,Image Data ID,Subject,Group,Sex,Age,Visit,MMSE Total Score,Global CDR,Acq Date,Image_name,filepath
1,I45109,002_S_0295,CN,M,85.4,sc,28.0,0.0,4/18/2006,002_S_0295__I45109,/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_...
6,I45118,002_S_0413,CN,F,76.9,sc,29.0,0.0,5/02/2006,002_S_0413__I45118,/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_...
11,I40675,002_S_0559,CN,M,79.9,sc,29.0,0.0,5/23/2006,002_S_0559__I40675,/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_...
16,I40684,002_S_0685,CN,F,89.7,sc,30.0,0.0,7/06/2006,002_S_0685__I40684,/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_...
34,I40732,002_S_0816,AD,M,72.3,sc,NaN,NaN,8/30/2006,002_S_0816__I40732,/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_...


In [73]:
df["filepath"][1]

'/opt/home/s4043685/Project1/ADNI/MNI152lin_T1_1mm_brain/Processed/002_S_0295__I45109.nii.gz'

In [29]:
df.to_csv(f"{data_path}/scan_info_new.csv", index = False)

In [30]:
# Counts
gender_counts = df.groupby(["Group", "Sex"]).size().reset_index(name = "Count")
print(gender_counts)

  Group Sex  Count
0    AD   F     74
1    AD   M     71
2    CN   F     90
3    CN   M     78


In [31]:
average_age = df.groupby(["Group", "Sex"])["Age"].mean().reset_index()
print(average_age)

  Group Sex        Age
0    AD   F  75.654054
1    AD   M  77.263380
2    CN   F  77.136667
3    CN   M  76.701282


In [32]:
mmse_score = df.groupby(["Group"])["MMSE Total Score"].mean().reset_index()
print(mmse_score)

  Group  MMSE Total Score
0    AD         22.541667
1    CN         29.131737


# PPMI

In [3]:
data_path = Settings["input_path"]
folder_name = 'MNI152lin_T1_1mm_brain'

In [4]:
data = pd.read_csv(f"{data_path}/zipped_1.5T_T1/PPMI_9_12_2025.csv")
data1 = pd.read_csv(f"{data_path}/zipped_3T_T1/PPMI_3T_10_05_2025.csv")

In [5]:
print(data.shape, data1.shape)

(212, 12) (1666, 12)


In [8]:
data = pd.concat([data, data1], ignore_index = True)

In [11]:
data["Group"].value_counts()

Group
PD         749
Control    213
Name: count, dtype: int64

In [10]:
data = data[data["Visit"] == "BL"]

In [12]:
data["Subject"] = data["Subject"].astype(str)

In [13]:
data["Image_name"] = data["Subject"] + "__" + data["Image Data ID"]
data["filepath"] = data_path + '/' + folder_name + "/Processed/" + data["Image_name"] + ".nii.gz"

In [14]:
data.head()

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Type,Acq Date,Format,Downloaded,Image_name,filepath
0,I223766,3051,PD,M,72,BL,MRI,SAG 3D T1,Original,10/26/2010,DCM,Yes,3051__I223766,/opt/home/s4043685/Project1/PPMI/MNI152lin_T1_...
1,I223769,3052,PD,F,66,BL,MRI,SAG 3D T1,Original,12/09/2010,DCM,Yes,3052__I223769,/opt/home/s4043685/Project1/PPMI/MNI152lin_T1_...
2,I223774,3053,Control,M,69,BL,MRI,SAG 3D T1,Original,12/09/2010,DCM,Yes,3053__I223774,/opt/home/s4043685/Project1/PPMI/MNI152lin_T1_...
3,I223782,3054,PD,M,74,BL,MRI,SAG 3D T1,Original,12/03/2010,DCM,Yes,3054__I223782,/opt/home/s4043685/Project1/PPMI/MNI152lin_T1_...
4,I223786,3055,Control,F,69,BL,MRI,SAG 3D T1,Original,12/03/2010,DCM,Yes,3055__I223786,/opt/home/s4043685/Project1/PPMI/MNI152lin_T1_...


In [15]:
average_age = data.groupby(["Group", "Sex"])["Age"].mean().reset_index()
print(average_age)

     Group Sex        Age
0  Control   F  59.941176
1  Control   M  62.453125
2       PD   F  63.530201
3       PD   M  62.662971


In [16]:
data.to_csv(f"{data_path}/scan_info.csv", index = False)